# RFID-ExSim 데이터 자율탐구 노트북

**2026 SIMs 1학년 자율탐구 프로젝트 · 데이터 기반 자율탐구**
데이터 출처: [RFID-ExSim-Dataset v1.0](https://github.com/yasserhmimou9/RFID-ExSim-Dataset/tree/v1.0) (Yasser Hmimou, MIT License)

이 노트북은 **자율탐구 로드맵** 순서를 따라갑니다.

| 파트 | 로드맵 단계 | 하는 일 | 보고서 칸 |
|---|---|---|---|
| 0 | – | 준비, 데이터 올리기 | 부록 B |
| 1 | ② 데이터 이해 | S1~S5가 무엇인지, 무엇을 알 수 **없는지** 확인 | Ⅱ-1, Ⅱ-2 |
| 2 | ③ 여러 방향 탐색 | S1~S5 공통 비교 + 시나리오별 그래프 | Ⅱ-3, Ⅱ-4, Ⅲ-1, Ⅲ-2 |
| 3 | ④ 패턴 고르기 · ⑤ 질문 | 가장 흥미로운 패턴 하나 고르기 | Ⅲ-4, Ⅳ-4 |
| 4 | ⑦ 가설 · ⑧ 의심하기 | 고른 패턴을 깊게 파고들기 | Ⅲ-3, Ⅳ-3, Ⅴ |
| 5 | ⑨ 다음 검증 | 가설·대안 설명·검증 계획 정리 | Ⅴ-1~4 |

**사용법**
- 위에서부터 차례로 실행하세요. (`Shift + Enter`, 또는 메뉴 `런타임 > 모두 실행`)
- ✏️ 표시된 칸은 **더블클릭해서 모둠이 직접 채우는 탐색 기록**입니다. 그래프를 많이 그리는 것보다 이 기록이 더 중요해요.
- 그래프는 `outputs/figures/` 폴더에 자동 저장됩니다. 마지막 셀에서 한 번에 내려받을 수 있어요.

## 0. 준비

In [ ]:
# 그래프에 한글이 보이도록 폰트 패키지 설치 (처음 한 번만, 약 10초)
%pip install -q koreanize-matplotlib

In [ ]:
import csv
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import koreanize_matplotlib  # noqa: F401  import만 하면 한글 폰트 적용
except ImportError:
    print("한글 폰트 패키지가 없어 글자가 네모로 보일 수 있어요.")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.unicode_minus": False,
})
pd.set_option("display.max_columns", 30)

SCENARIOS = ["S1", "S2", "S3", "S4", "S5"]
SC_NAME = {"S1": "S1 정상", "S2": "S2 동시읽기", "S3": "S3 복제", "S4": "S4 재전송", "S5": "S5 폭주"}
# 항상 같은 대상에는 같은 색 (색약자도 구분 가능한 팔레트)
SC_COLOR = {"S1": "#2a78d6", "S2": "#eb6834", "S3": "#1baf7a", "S4": "#eda100", "S5": "#e87ba4"}
ANGLE_COLOR = {0: "#2a78d6", 45: "#eb6834", 90: "#1baf7a"}
READER_COLOR = {"ESP32_A": "#2a78d6", "ESP32_B": "#eb6834"}
DIST_COLOR = {1: "#9cc3f0", 2: "#5f9be0", 3: "#2a78d6", 4: "#174a8c"}  # 가까울수록 연하게

FIG_DIR = Path("outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    # 보고서에 붙일 수 있도록 그림을 PNG로 저장
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

### 데이터 올리기
- **Colab**: 아래 셀을 실행하면 파일 선택 창이 뜹니다. 받아 둔 `RFID-ExSim-dataset.zip`을 고르세요. (약 6MB)
- **Google Drive에 넣어 두었다면**: 왼쪽 폴더 아이콘 → Drive 연결 후, 아래 `ZIP_PATH`를 그 경로로 바꾸세요.
- 이미 압축이 풀려 있으면 업로드 없이 넘어갑니다.

In [ ]:
ZIP_PATH = Path("RFID-ExSim-dataset.zip")   # Drive를 쓰면 예: Path("/content/drive/MyDrive/RFID-ExSim-dataset.zip")

def find_processed():
    hits = [p for p in Path(".").rglob("rfid_dataset_S1_all.jsonl") if "__MACOSX" not in p.parts]
    return hits[0].parent if hits else None

if find_processed() is None:
    if not ZIP_PATH.exists():
        from google.colab import files  # Colab에서만 동작
        uploaded = files.upload()
        ZIP_PATH = Path(next(name for name in uploaded if name.endswith(".zip")))
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("data_unzipped")

PROC = find_processed()                # data/processed 폴더
PLANS = PROC.parent / "plans"          # data/plans 폴더
print("processed 폴더:", PROC)
print("plans 폴더:", PLANS)

In [ ]:
# S1~S5 기록 파일 읽기 (S5가 커서 30초 정도 걸릴 수 있어요)
data = {}
for sc in SCENARIOS:
    d = pd.read_json(PROC / f"rfid_dataset_{sc}_all.jsonl", lines=True, dtype=False)
    d["ts"] = pd.to_datetime(d["timestamp_iso"], format="ISO8601", utc=True)
    data[sc] = d
    print(f"{sc}: {len(d):>8,}줄")

# 요약 파일과 실험 계획표
meta1 = pd.read_csv(PROC / "meta_sessions_S1.csv")
meta2 = pd.read_csv(PROC / "meta_sessions_S2.csv")
s2_pairs = pd.read_csv(PROC / "s2_collisions_summary.csv")
s3_pairs = pd.read_csv(PROC / "s3_clone_collisions_summary.csv")
s5_rate = pd.read_csv(PROC / "s5_success_rates_per_session.csv")
plan4 = pd.read_csv(PLANS / "s4_plan.csv")

# s5_plan.csv는 마지막 notes 칸 안에 쉼표가 있어 일반 방법으로 읽으면 칸이 밀려요 → 직접 읽기
with open(PLANS / "s5_plan.csv", encoding="utf-8") as f:
    rows = list(csv.reader(f))
header = rows[0]
plan5 = pd.DataFrame([r[:len(header) - 1] + [",".join(r[len(header) - 1:])] for r in rows[1:]], columns=header)
for col in ["distance_cm", "orientation_deg", "burst_size", "burst_rate_hz", "bursts"]:
    plan5[col] = pd.to_numeric(plan5[col])

---
# 파트 1. 데이터를 제대로 이해하기 `로드맵 ②` `보고서 Ⅱ-1, Ⅱ-2`

분석하기 전에 **한 행(row)이 무엇인지, 변수는 무엇인지, 무엇을 알 수 없는지**부터 확인합니다.
여기서 **한 행 = 리더기가 카드를 한 번 읽은(또는 읽으려 한) 기록 1건**이에요.

In [ ]:
def values(d, col):
    if col not in d:
        return "열 없음"
    v = sorted(d[col].dropna().unique())
    return ", ".join(f"{x:g}" for x in v) if v else "비어 있음"

overview = pd.DataFrame([{
    "시나리오": SC_NAME[sc],
    "기록 수": f"{len(d):,}",
    "세션 수": d["session_id"].nunique(),
    "리더기": ", ".join(sorted(d["device_id"].dropna().unique())),
    "기록 종류(event)": ", ".join(f"{k} {v:,}" for k, v in d["event"].value_counts().items()),
    "거리(cm)": values(d, "distance_cm"),
    "각도(°)": values(d, "orientation_deg"),
    "측정 기간(UTC)": f"{d['ts'].min():%m/%d} ~ {d['ts'].max():%m/%d}",
} for sc, d in data.items()]).set_index("시나리오")
overview

**열마다 값이 채워진 비율(%)**: `-`는 그 시나리오 파일에 아예 없는 열이에요. 결측치를 확인할 때 씁니다.

In [ ]:
filled = pd.DataFrame({SC_NAME[sc]: d.drop(columns="ts").notna().mean() * 100 for sc, d in data.items()})
filled.round(1).fillna("-")

In [ ]:
# 이 데이터로 '알 수 없는 것' 찾기: 결과를 보고 보고서 Ⅱ-1 '데이터의 한계'에 적으세요
print("S1에 읽기 실패(read_fail) 기록이 있나?     ", (data["S1"]["event"] == "read_fail").sum(), "건")
print("S1의 raw 기록은 무엇인가? (상위 3개)")
print(data["S1"].query("event == 'raw'")["raw_payload"].value_counts().head(3).to_string(), "\n")
print("S2 요약 파일의 신호 세기(mean_rssi) 값 종류:", meta2["mean_rssi"].unique())
print("S5 기록 파일에 거리·각도 열이 있나?       ", {"distance_cm", "orientation_deg"} <= set(data["S5"].columns))
print("공개 전에 지운다고 한 원본 UID(uid_plain_local)가 남아 있나?",
      {sc: "uid_plain_local" in d for sc, d in data.items()})

> ✏️ **탐색 기록: 데이터 이해 (보고서 Ⅱ-1 변수 표, Ⅱ-2 품질 점검)** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

---
# 파트 2. 여러 방향에서 탐색하기 `로드맵 ③` `보고서 Ⅱ-4, Ⅲ-1, Ⅲ-2`

## 2-1. 공통 비교: 거리·각도에 따라 성공률이 달라질까? (집단 비교)

다섯 시나리오를 **같은 모양의 그래프**로 나란히 놓고 비교합니다.
단, 시나리오마다 '성공률'을 계산하는 방법이 다르다는 점을 꼭 기억하세요. (보고서 Ⅱ-3에 적을 것)

| 시나리오 | 이 노트북에서 쓴 '성공률' |
|---|---|
| S1 | 요약 파일(`meta_sessions_S1.csv`)의 success_rate |
| S2 | 요약 파일(`meta_sessions_S2.csv`)의 success_rate |
| S3 | 전체 기록 중 `read_success` 비율 |
| S4 | 전체 기록 중 `read_fail`이 **아닌** 비율 |
| S5 | 요약 파일(`s5_success_rates_per_session.csv`)의 success_rate_pct (= B가 받은 수 ÷ A가 보낸 수) |

In [ ]:
def by_event(d, ok):
    return (d.assign(ok=ok(d))
              .groupby(["distance_cm", "orientation_deg"])["ok"].mean().mul(100)
              .rename("success").reset_index())

s5_sessions = plan5.merge(s5_rate, on="session_id")
cond = pd.concat([
    meta1.groupby(["distance_cm", "orientation_deg"])["success_rate"].mean().rename("success").reset_index().assign(scenario="S1"),
    meta2.groupby(["distance_cm", "orientation_deg"])["success_rate"].mean().rename("success").reset_index().assign(scenario="S2"),
    by_event(data["S3"], lambda d: d["event"] == "read_success").assign(scenario="S3"),
    by_event(data["S4"], lambda d: d["event"] != "read_fail").assign(scenario="S4"),
    s5_sessions.groupby(["distance_cm", "orientation_deg"])["success_rate_pct"].mean().rename("success").reset_index().assign(scenario="S5"),
], ignore_index=True)
cond["distance_cm"] = cond["distance_cm"].astype(int)
cond["orientation_deg"] = cond["orientation_deg"].astype(int)

fig, axes = plt.subplots(1, 5, figsize=(17, 4), sharey=True)
for ax, sc in zip(axes, SCENARIOS):
    part = cond[cond["scenario"] == sc]
    for dist, g in part.groupby("distance_cm"):
        g = g.sort_values("orientation_deg")
        ax.plot(g["orientation_deg"], g["success"], marker="o", ms=7, lw=2,
                color=DIST_COLOR[dist], label=f"{dist}cm")
    ax.set_title(SC_NAME[sc])
    ax.set_xticks([0, 45, 90], ["0°", "45°", "90°"])
    ax.set_xlabel("카드 각도")
    ax.set_ylim(0, 105)
    ax.legend(title="거리", fontsize=8, title_fontsize=8, loc="lower left")
axes[0].set_ylabel("성공률 (%)")
fig.suptitle("시나리오별 · 거리·각도별 성공률", fontsize=14)
plt.tight_layout()
save(fig, "2-1_시나리오별_거리각도_성공률")
plt.show()

> ✏️ **탐색 기록: 2-1 공통 비교: 어느 시나리오에서 선이 평평하고, 어디서 기울어졌나?** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

## 2-2. 시나리오별로 한 가지씩 더 보기

### S1 정상: 읽기 간격은 어떻게 분포할까? (분포)
**읽기 간격** = 같은 세션에서 바로 앞 읽기와 지금 읽기 사이의 시간입니다.

In [ ]:
s1 = data["S1"].query("event == 'read_success'").sort_values("ts").copy()
s1["interval_s"] = s1.groupby(["session_id", "device_id"])["ts"].diff().dt.total_seconds()
iv = s1["interval_s"].dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(iv.clip(upper=5), bins=np.arange(0, 5.05, 0.05), color=SC_COLOR["S1"], edgecolor="white", linewidth=0.5)
ax.axvline(iv.min(), color="black", ls="--", lw=1)
ax.annotate(f"가장 짧은 간격 {iv.min():.3f}초", (iv.min(), ax.get_ylim()[1] * 0.9), xytext=(-6, 0), textcoords="offset points", ha="right")
ax.set_xlabel("읽기 간격 (초, 5초 이상은 5에 모음)")
ax.set_ylabel("횟수")
ax.set_title("S1: 정상 읽기의 간격 분포")
save(fig, "2-2_S1_읽기간격_분포")
plt.show()
print(iv.describe().round(3).to_string())

> ✏️ **탐색 기록: S1 읽기 간격** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

### S2 동시읽기: 두 리더기는 얼마나 차이 나게 읽을까? (분포)

In [ ]:
dt_ms = s2_pairs["dt"] * 1000   # 리더 A 시각 − 리더 B 시각 (ms)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(dt_ms.clip(-400, 400), bins=80, color=SC_COLOR["S2"], edgecolor="white", linewidth=0.5)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("A가 읽은 시각 - B가 읽은 시각 (ms, ±400 밖은 양 끝에 모음)   ← A가 먼저 | B가 먼저 →")
ax.set_ylabel("횟수")
ax.set_title("S2: 두 리더기 읽기 시각의 차이")
save(fig, "2-2_S2_리더시간차")
plt.show()
print("A·B 읽기 수:", data["S2"]["device_id"].value_counts().to_dict())
print(f"|시간차| 중앙값: {dt_ms.abs().median():.1f} ms")

> ✏️ **탐색 기록: S2 두 리더기의 시간차** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

### S3 복제: 같은 UID가 두 리더에 얼마나 동시에 찍힐까? (분포 · 이상치)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
vc = s3_pairs["delta_ms"].round().astype(int).value_counts().sort_index()
ax.bar(vc.index, vc.values, color=SC_COLOR["S3"], edgecolor="white", width=0.7)
ax.set_xlabel("두 리더 기록의 시간차 (ms)")
ax.set_ylabel("횟수")
ax.set_title("S3: 복제 카드 쌍의 시간차")
save(fig, "2-2_S3_복제_시간차")
plt.show()
print("반올림한 시간차 값의 종류:", sorted(s3_pairs["delta_ms"].round().astype(int).unique()), "ms")

# 세션·리더마다 기록이 몇 건씩 있나? (실제 측정이라면 조금씩 달라야 자연스러움)
per_session = data["S3"].groupby(["session_id", "device_id"]).size()
print("세션·리더 하나당 기록 수:", per_session.value_counts().to_dict())

> ✏️ **탐색 기록: S3 복제 (숫자가 너무 고르지 않은가?)** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

### S4 재전송: 기록 종류는 공격 방식마다 어떻게 다를까? (집단 비교)

In [ ]:
s4 = data["S4"].merge(plan4[["session_id", "replay_type"]], on="session_id", how="left")
comp = pd.crosstab(s4["replay_type"], s4["event"], normalize="index").mul(100)
comp = comp[[c for c in ["read_success", "read_injected", "read_fail"] if c in comp]]
EVENT_COLOR = {"read_success": "#2a78d6", "read_injected": "#eb6834", "read_fail": "#1baf7a"}

fig, ax = plt.subplots(figsize=(10, 3.5))
left = np.zeros(len(comp))
for col in comp.columns:
    ax.barh(comp.index, comp[col], left=left, color=EVENT_COLOR[col], edgecolor="white", linewidth=2, label=col)
    for y, (l, w) in enumerate(zip(left, comp[col])):
        if w >= 5:
            ax.text(l + w / 2, y, f"{w:.1f}%", ha="center", va="center", color="white", fontsize=9)
    left += comp[col].values
ax.set_xlabel("기록 비율 (%)")
ax.set_title("S4: 재전송 방식별 기록 종류")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3)
ax.grid(False)
save(fig, "2-2_S4_기록종류")
plt.show()
display(comp.round(2))

> ✏️ **탐색 기록: S4 재전송 방식별 기록 종류** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

### S5 폭주: 공격 중 읽기 수는 시간에 따라 어떻게 변할까? (시간 변화)

In [ ]:
SID = "S5_TAG01_run1"   # 다른 세션도 바꿔서 보세요 (예: S5_TAG03_run3)
g = data["S5"].query("session_id == @SID")
t0 = g["ts"].min()

fig, ax = plt.subplots(figsize=(12, 4))
for dev, gg in g.groupby("device_id"):
    per = gg.set_index("ts").resample("100ms").size()
    ax.plot((per.index - t0).total_seconds(), per.values * 10, lw=1.5, color=READER_COLOR[dev], label=dev)
ax.set_xlabel("세션 시작 후 경과 시간 (초)")
ax.set_ylabel("초당 기록 수 (0.1초 단위로 셈)")
ax.set_title(f"S5: {SID} 기록 수의 시간 변화")
ax.legend(title="리더기")
save(fig, f"2-2_S5_시계열_{SID}")
plt.show()
display(s5_sessions.query("session_id == @SID")[["session_id", "burst_size", "burst_rate_hz", "bursts", "reads_A", "reads_B", "success_rate_pct"]])

> ✏️ **탐색 기록: S5 폭주 시계열 (A와 B의 역할은 무엇일까?)** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

---
# 파트 3. 흥미로운 패턴 하나 고르기 `로드맵 ④ ⑤` `보고서 Ⅲ-4, Ⅳ-4`

2-1의 그래프를 숫자 하나로 요약해 봅니다.
**"거리(또는 각도)를 바꾸면 성공률이 최대 몇 %p 달라지나?"**

In [ ]:
def spread(col):
    m = cond.groupby(["scenario", col])["success"].mean()
    return m.groupby("scenario").agg(lambda s: s.max() - s.min())

effect = pd.DataFrame({"거리를 바꿀 때": spread("distance_cm"), "각도를 바꿀 때": spread("orientation_deg")}).reindex(SCENARIOS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, col in zip(axes, effect.columns):
    bars = ax.bar([SC_NAME[s] for s in effect.index], effect[col], color=[SC_COLOR[s] for s in effect.index], edgecolor="white")
    ax.bar_label(bars, fmt="%.1f", padding=2)
    ax.set_title(f"{col} 성공률이 달라지는 폭")
axes[0].set_ylabel("최대 - 최소 (%p)")
plt.tight_layout()
save(fig, "3_거리각도_효과크기")
plt.show()
effect.round(2)

> ✏️ **로드맵 ④ 내가 집중해서 탐구할 패턴** (더블클릭해서 작성)
> - 처음 예상과 다른 결과는?
> - 조건에 따라 크게 달라지는 결과는?
> - 두 가지 이상의 설명이 가능한 결과는?
> - **고른 패턴**:

> ✏️ **로드맵 ⑤ 탐구 질문으로 바꾸기**
> - 너무 넓은 질문 예: "각도는 RFID에 영향을 줄까?"
> - **현재 탐구 질문** (어떤 변수? 어떤 조건? 이 데이터로 답할 수 있나?):

---
# 파트 4. 고른 패턴 깊게 파기: S1(기준) + S5(폭주) `로드맵 ⑦ ⑧` `보고서 Ⅲ-3, Ⅳ-3`

## 4-1. 기준선 점검: S1의 '성공률'은 무엇을 센 값일까?
S1을 비교 기준으로 쓰려면, 그 성공률이 믿을 만한지부터 확인해야 해요.
S1 요약 파일의 숫자를 원본 기록과 맞춰 봅니다.

In [ ]:
raw_lines = data["S1"].query("event == 'raw'").groupby("session_id").size()
m1 = meta1.assign(raw_lines=meta1["session_id"].map(raw_lines).fillna(0).astype(int))

print(f"세션 수: {len(m1)}")
print(f"전체 기록 수 = 성공 수 + raw(부팅 메시지) 수 인 세션: {(m1.total_reads == m1.success_reads + m1.raw_lines).mean():.1%}")
print(f"전체 기록 수 = 계획한 반복 수(repetitions) 인 세션:   {(m1.total_reads == m1.repetitions).mean():.1%}")
m1[["session_id", "repetitions", "total_reads", "success_reads", "raw_lines", "success_rate"]].sort_values("success_rate").head(8)

> ✏️ **탐색 기록: 4-1 S1 성공률은 무엇을 센 값인가?** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

## 4-2. S5에서는 각도만 바뀌었을까? 실험 계획표 확인하기
S5의 각도별 성공률과 **각도별 초당 요청 수(burst_rate_hz)**를 나란히 봅니다.

In [ ]:
angles = [0, 45, 90]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col, label in [(axes[0], "success_rate_pct", "성공률 (%)"), (axes[1], "burst_rate_hz", "초당 요청 수 (회/초)")]:
    for a in angles:
        y = s5_sessions.loc[s5_sessions["orientation_deg"] == a, col]
        x = np.full(len(y), a) + np.random.default_rng(0).uniform(-4, 4, len(y))
        ax.scatter(x, y, s=60, color=ANGLE_COLOR[a], edgecolor="white", linewidth=1.5, zorder=3)
        ax.hlines(y.mean(), a - 10, a + 10, color="black", lw=2, zorder=4)
    ax.set_xticks(angles, ["0°", "45°", "90°"])
    ax.set_xlabel("카드 각도")
    ax.set_ylabel(label)
axes[0].set_title("S5: 각도별 성공률 (검은 선 = 평균)")
axes[1].set_title("S5: 각도별 초당 요청 수 (검은 선 = 평균)")
plt.tight_layout()
save(fig, "4-2_S5_각도별_성공률과_요청속도")
plt.show()

print("요청 속도 × 각도별 세션 수 (같은 속도에 여러 각도가 있는 행이 몇 개인가?)")
pd.crosstab(s5_sessions["burst_rate_hz"], s5_sessions["orientation_deg"])

> ✏️ **탐색 기록: 4-2 각도와 함께 바뀐 다른 변수는?** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

## 4-3. 요청 속도와 성공률의 관계 (변수 관계 · 모델)

In [ ]:
x = s5_sessions["burst_rate_hz"]
y = s5_sessions["success_rate_pct"]
slope, intercept = np.polyfit(x, y, 1)
r = np.corrcoef(x, y)[0, 1]

fig, ax = plt.subplots(figsize=(10, 5))
for a in angles:
    part = s5_sessions[s5_sessions["orientation_deg"] == a]
    ax.scatter(part["burst_rate_hz"], part["success_rate_pct"], s=70, color=ANGLE_COLOR[a],
               edgecolor="white", linewidth=1.5, label=f"{a}°", zorder=3)
xs = np.linspace(x.min(), x.max(), 50)
ax.plot(xs, slope * xs + intercept, color="black", lw=1.5, ls="--", zorder=2)
ax.annotate(f"성공률 ≈ {intercept:.1f} {slope:+.3f} × 요청속도\n상관계수 r = {r:.3f},  $R^2$ = {r**2:.3f}",
            (0.98, 0.95), xycoords="axes fraction", ha="right", va="top")
ax.set_xlabel("초당 요청 수 (회/초)")
ax.set_ylabel("성공률 (%)")
ax.set_title("S5: 요청 속도가 빠를수록 성공률은?")
ax.legend(title="각도")
save(fig, "4-3_S5_요청속도_성공률")
plt.show()
print(f"요청 속도가 100회/초 늘 때마다 성공률은 약 {abs(slope) * 100:.1f}%p 변함")
print(f"이 식을 믿을 수 있는 범위(예측 범위): {x.min()} ~ {x.max()} 회/초")

> ✏️ **탐색 기록: 4-3 요청 속도와 성공률 (모델이 맞는 범위는?)** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

## 4-4. 같은 요청 속도에서 각도만 다르면? (조건을 맞춰 비교)
요청 속도를 같게 맞춘 세션끼리만 비교하면 **각도의 영향만** 따로 볼 수 있어요.
그런 세션이 몇 개나 있는지도 꼭 세어 보세요.

In [ ]:
multi = s5_sessions.groupby("burst_rate_hz")["orientation_deg"].nunique()
same_rate = s5_sessions[s5_sessions["burst_rate_hz"].isin(multi[multi > 1].index)].sort_values(["burst_rate_hz", "orientation_deg"])

fig, ax = plt.subplots(figsize=(8, 4))
for rate, g in same_rate.groupby("burst_rate_hz"):
    ax.plot(g["orientation_deg"], g["success_rate_pct"], color="lightgray", lw=1, zorder=1)
    for _, row in g.iterrows():
        ax.scatter(row["orientation_deg"], row["success_rate_pct"], s=80, color=ANGLE_COLOR[int(row["orientation_deg"])],
                   edgecolor="white", linewidth=1.5, zorder=3)
    ax.annotate(f"{rate}회/초", (g["orientation_deg"].iloc[-1], g["success_rate_pct"].iloc[-1]), xytext=(8, 0), textcoords="offset points", va="center")
ax.set_xticks(angles, ["0°", "45°", "90°"])
ax.set_xlim(-15, 110)
ax.set_xlabel("카드 각도")
ax.set_ylabel("성공률 (%)")
ax.set_title("S5: 요청 속도가 같은 세션끼리 각도 비교")
save(fig, "4-4_S5_같은속도_각도비교")
plt.show()
print("비교에 쓸 수 있는 세션 수:", len(same_rate))
same_rate[["session_id", "burst_rate_hz", "orientation_deg", "distance_cm", "success_rate_pct"]].round(1)

> ✏️ **탐색 기록: 4-4 속도를 맞추면 각도 차이는? (세션 수가 충분한가?)** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

## 4-5. 내 결과를 의심해 보기: 데이터 수집 방식 때문에 생긴 결과는 아닐까? `로드맵 ⑧`
4-3의 상관계수가 **실제 측정치고 너무 완벽하지 않은지** 의심해 봅니다.
리더기 A의 기록 간격이 계획표의 요청 속도와 얼마나 정확히 일치하는지 확인해요.

In [ ]:
s5 = data["S5"].sort_values("ts")
a_iv = (s5.query("device_id == 'ESP32_A'")
          .assign(iv_ms=lambda d: d.groupby("session_id")["ts"].diff().dt.total_seconds() * 1000)
          .dropna(subset=["iv_ms"]))
check = (a_iv.groupby("session_id")["iv_ms"]
             .agg(가장흔한간격_ms=lambda s: s.round(3).mode()[0], 중앙값_ms="median")
             .join(s5_sessions.set_index("session_id")[["burst_rate_hz", "reads_A", "burst_size", "bursts"]]))
check["계획상_간격_ms"] = (1000 / check["burst_rate_hz"]).round(3)
check["계획 요청 수"] = check["burst_size"] * check["bursts"]
print("리더 A 기록 수 = 계획한 요청 수 인 세션 비율:", f"{(check['reads_A'] == check['계획 요청 수']).mean():.0%}")
check[["burst_rate_hz", "계획상_간격_ms", "가장흔한간격_ms", "중앙값_ms", "reads_A", "계획 요청 수"]].head(10)

> ✏️ **탐색 기록: 4-5 데이터가 실제 측정인지, 프로그램이 만든 것인지 판단할 근거는?** (더블클릭해서 직접 채우세요)
> - **본 사실** (그래프에서 직접 보이는 것):
> - **가능한 해석** (이것이 의미할 수 있는 것):
> - **새 질문** (그래서 무엇이 더 궁금한가?):

---
# 파트 5. 가설 · 대안 설명 · 다음 검증 정리 `로드맵 ⑦ ⑧ ⑨` `보고서 Ⅴ`

> ⚠️ 같은 데이터에서 찾은 패턴으로 만든 가설은 그 데이터만으로 **"검증되었다"고 말할 수 없습니다.**
> 앞으로 어떻게 검토할지 계획을 세우는 것이 목표예요.

> ✏️ **⑦ 탐색적 가설** (보고서 Ⅴ-1)
> - 데이터에서 본 사실:
> - 관련 자료에서 알게 된 내용 (최소 2개, 보고서 Ⅳ-2·Ⅷ):
> - 가능한 설명 1:
> - 가능한 설명 2:
> - **나의 탐색적 가설** ("이 데이터 범위에서", "주로" 같은 범위 표현 넣기):

> ✏️ **가설의 근거** (보고서 Ⅴ-2 표)
> | 근거 요소 | 내용 |
> |---|---|
> | 데이터 근거 | (파트 2-1, 4-2, 4-3, 4-4의 숫자) |
> | 이론·선행 자료 근거 | |
> | 대안 설명과의 비교 | |

> ✏️ **⑧ 대안 설명** (보고서 Ⅳ-3)
> - 대안 설명 1: (4-4: 비교할 수 있는 세션이 적다면?)
> - 대안 설명 2: (4-1, 4-5: 데이터 수집 방식 때문이라면?)

> ✏️ **⑨ 다음 검증 계획** (보고서 Ⅴ-4)
> - 어떤 조건을 어떻게 나눠야 각도와 요청 속도의 영향을 분리할 수 있을까? (예: 각도 3개 × 요청 속도 3개 × 반복 ○회)
> - 추가로 필요한 데이터:
> - 다음에 할 분석 또는 실험:

## 결과 내려받기 `보고서 부록 A`

In [ ]:
# 저장된 그래프 목록
for p in sorted(FIG_DIR.glob("*.png")):
    print(p.name)

# Colab이면 그래프 폴더를 zip으로 묶어 내려받기
import shutil
shutil.make_archive("rfid_figures", "zip", FIG_DIR)
try:
    from google.colab import files
    files.download("rfid_figures.zip")
except ImportError:
    print("Colab이 아니면 outputs/figures 폴더에서 바로 꺼내 쓰면 됩니다.")